# 🎯 HiringCafe Job Matching Application

This notebook scrapes job listings from HiringCafe, reads the user's CV, and analyzes whether each job is a good match using two different LLMs.

**Features:**
- 📄 Automatic CV parsing from PDF or TXT files
- 🌐 Web scraping job listings from HiringCafe
- 🤖 Job matching analysis with OpenAI GPT-4o-mini
- 🦙 Job matching analysis with Ollama (qwen3:4b)
- 📊 Side-by-side comparison report of both models

## 1. Imports & Setup

In [1]:
import os
import re
import json
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, HTML

# Load environment variables
load_dotenv(override=True)

# OpenAI client
openai_client = OpenAI()

# Ollama client (OpenAI-compatible API)
ollama_base_url = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
ollama_client = OpenAI(base_url=ollama_base_url, api_key='ollama')

# HTTP headers for web scraping
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/117.0.0.0 Safari/537.36'
}

print("Setup complete!")
print(f"   OpenAI API: {'Active' if os.getenv('OPENAI_API_KEY') else 'Key not found'}")
print(f"   Ollama URL: {ollama_base_url}")

Setup complete!
   OpenAI API: Active
   Ollama URL: http://localhost:11434/v1


## 2. CV Reading Function

Reads your CV from a PDF or TXT file and returns it as plain text.

In [2]:
def read_cv(file_path: str) -> str:
    """Read a CV file (supports PDF and TXT formats)."""
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"CV file not found: {file_path}")

    if path.suffix.lower() == '.pdf':
        from pypdf import PdfReader
        reader = PdfReader(file_path)
        text_parts = []
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text_parts.append(page_text)
        cv_text = "\n".join(text_parts)
    elif path.suffix.lower() in ('.txt', '.md'):
        with open(file_path, 'r', encoding='utf-8') as f:
            cv_text = f.read()
    else:
        raise ValueError(f"Unsupported file format: {path.suffix}. Use PDF or TXT.")

    if not cv_text.strip():
        raise ValueError("CV file is empty!")

    return cv_text.strip()


# ────────────────────────────────────────────────
# Set your CV file path below.
# Example: cv_path = "my_cv.pdf" or cv_path = "cv.txt"
# If no file is provided, a sample CV will be used.
# ────────────────────────────────────────────────

cv_path = None  # <-- Set your CV file path here

if cv_path and Path(cv_path).exists():
    cv_text = read_cv(cv_path)
    print(f"CV loaded: {cv_path} ({len(cv_text)} characters)")
else:
    # Sample CV text
    cv_text = """
    ALPEREN KURT
    Software Engineer | AI/ML Enthusiast
    Istanbul, Turkey

    SUMMARY
    Passionate software engineer with 3+ years of experience in backend development,
    machine learning, and cloud technologies. Strong interest in LLM applications,
    NLP, and AI-driven products.

    TECHNICAL SKILLS
    - Programming: Python, Java, JavaScript/TypeScript, SQL
    - AI/ML: PyTorch, TensorFlow, Hugging Face, LangChain, OpenAI API
    - Backend: FastAPI, Django, Spring Boot, REST APIs
    - Cloud: AWS (EC2, S3, Lambda), Docker, Kubernetes
    - Databases: PostgreSQL, MongoDB, Redis
    - Tools: Git, CI/CD, Linux, Selenium

    EXPERIENCE
    Software Engineer | XYZ Tech (2022 - Present)
    - Developed and maintained microservices using Python/FastAPI
    - Built ML pipelines for text classification and NER tasks
    - Deployed models to AWS using Docker and Kubernetes
    - Integrated OpenAI GPT APIs into production applications

    Junior Developer | ABC Corp (2020 - 2022)
    - Built RESTful APIs using Java/Spring Boot
    - Worked with PostgreSQL and MongoDB databases
    - Implemented automated testing with JUnit and pytest

    EDUCATION
    B.Sc. Computer Engineering | Istanbul Technical University (2020)

    LANGUAGES
    - Turkish (Native)
    - English (Professional)
    - German (Basic - A2)
    """.strip()
    print("No CV file specified, using sample CV.")
    print("To use your own CV, update the cv_path variable above.")

print(f"\nCV Preview (first 500 chars):\n{'─' * 50}")
print(cv_text[:500])

No CV file specified, using sample CV.
To use your own CV, update the cv_path variable above.

CV Preview (first 500 chars):
──────────────────────────────────────────────────
ALPEREN KURT
    Software Engineer | AI/ML Enthusiast
    Istanbul, Turkey

    SUMMARY
    Passionate software engineer with 3+ years of experience in backend development,
    machine learning, and cloud technologies. Strong interest in LLM applications,
    NLP, and AI-driven products.

    TECHNICAL SKILLS
    - Programming: Python, Java, JavaScript/TypeScript, SQL
    - AI/ML: PyTorch, TensorFlow, Hugging Face, LangChain, OpenAI API
    - Backend: FastAPI, Django, Spring Boot, REST APIs
    


## 3. HiringCafe Scraping Functions

Fetches job listing links and their details from HiringCafe pages.

In [3]:
def fetch_job_links(category_url: str, max_jobs: int = 5) -> list[dict]:
    """
    Scrape job listing links from a HiringCafe category/search page.

    Args:
        category_url: HiringCafe category URL (e.g. https://hiringcafe.com/jobs/software-engineer)
        max_jobs: Maximum number of listings to fetch

    Returns:
        List of dicts: [{"title": ..., "url": ...}, ...]
    """
    print(f"Scanning listings: {category_url}")

    response = requests.get(category_url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    # Job listing links on HiringCafe start with /job/
    job_links = []
    seen_urls = set()

    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        # Capture links starting with /job/
        if '/job/' in href and href not in seen_urls:
            # Build full URL
            if href.startswith('/'):
                full_url = f"https://hiringcafe.com{href}"
            elif href.startswith('https://hiringcafe.com/job/'):
                full_url = href
            else:
                continue

            # Extract title from URL slug
            slug = href.split('/job/')[-1]
            title_parts = slug.rsplit('-', 1)[0] if '-' in slug else slug
            title = title_parts.replace('-', ' ').title()

            seen_urls.add(href)
            job_links.append({
                'title': title,
                'url': full_url
            })

            if len(job_links) >= max_jobs:
                break

    print(f"   Found {len(job_links)} listings")
    return job_links


def fetch_job_details(job_url: str) -> dict:
    """
    Extract details from a single job listing page.

    Returns:
        {"url": ..., "title": ..., "company": ..., "raw_text": ..., "sections": {...}}
    """
    response = requests.get(job_url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    # Title
    title = soup.title.string if soup.title else "Unknown"
    title = title.replace(' | HiringCafe', '').strip()

    # Company name - extract from title ("... at CompanyName" format)
    company = "Unknown"
    if ' at ' in title:
        company = title.split(' at ')[-1].strip()

    # Extract body text (remove nav, script, style, etc.)
    if soup.body:
        for tag in soup.body(['script', 'style', 'nav', 'footer', 'header', 'img', 'input', 'svg']):
            tag.decompose()
        raw_text = soup.body.get_text(separator='\n', strip=True)
    else:
        raw_text = ""

    # Find section headings (h2 tags)
    sections = {}
    for h2 in soup.find_all('h2'):
        section_title = h2.get_text(strip=True)
        content_parts = []
        for sibling in h2.find_next_siblings():
            if sibling.name == 'h2':
                break
            text = sibling.get_text(strip=True)
            if text:
                content_parts.append(text)
        if content_parts:
            sections[section_title] = '\n'.join(content_parts)

    return {
        'url': job_url,
        'title': title,
        'company': company,
        'raw_text': raw_text[:5000],  # Truncate for LLM context limit
        'sections': sections
    }


# Quick test: fetch details from a single listing
test_url = "https://hiringcafe.com/job/senior-backend-java-developer-m-w-d-envidual-munich-bavaria-xyi2cdxp7sslww3o"
test_job = fetch_job_details(test_url)
print(f"\nTest listing details:")
print(f"   Title: {test_job['title']}")
print(f"   Company: {test_job['company']}")
print(f"   Sections: {list(test_job['sections'].keys())}")
print(f"   Text length: {len(test_job['raw_text'])} chars")


Test listing details:
   Title: Senior Backend Java Developer (m/w/d) at Envidual
   Company: Envidual
   Sections: ['Senior Backend Java Developer (m/w/d)', 'Aufgaben', 'Qualifikation', 'Benefits']
   Text length: 3882 chars


## 4. LLM Prompts

System and user prompts for job matching analysis.

In [4]:
JOB_MATCHING_SYSTEM_PROMPT = """
You are an experienced career advisor and job matching specialist.
You will be given a candidate's CV and a job listing.

Your task:
1. Analyze the job listing requirements
2. Evaluate the candidate's skills and experience from their CV
3. Determine whether the candidate is a good match for this position

You MUST respond ONLY with a JSON object in the following format (no other text):
{
    "match_score": <number from 1 to 10>,
    "decision": "<Good Match | Partial Match | Not a Match>",
    "matching_skills": ["skill1", "skill2", ...],
    "missing_skills": ["skill1", "skill2", ...],
    "strengths": "<Brief explanation of the candidate's strongest qualifications for this role>",
    "weaknesses": "<Brief explanation of areas where the candidate falls short>",
    "overall_assessment": "<2-3 sentence detailed assessment>"
}

Important rules:
- Score from 1 to 10 (1=completely unqualified, 10=perfect match)
- Be objective, base your evaluation solely on the CV content
- List missing skills so the candidate knows where to improve
- Respond in English
"""


def build_matching_user_prompt(cv_text: str, job_info: dict) -> str:
    """Build the user prompt combining CV and job listing information."""

    # Format section information
    sections_text = ""
    for section_title, content in job_info.get('sections', {}).items():
        sections_text += f"\n### {section_title}\n{content}\n"

    return f"""## CANDIDATE'S CV:
{cv_text}

---

## JOB LISTING DETAILS:
- **Position:** {job_info['title']}
- **Company:** {job_info['company']}
- **URL:** {job_info['url']}

{sections_text}

### Raw Listing Text:
{job_info['raw_text'][:3000]}

---

Based on the CV and job listing above, perform the matching analysis and return JSON.
"""

print("Prompts defined successfully.")

Prompts defined successfully.


## 5. LLM Matching Functions

Functions that perform the matching analysis using both OpenAI and Ollama.

In [5]:
def parse_llm_response(raw_response: str) -> dict:
    """Parse JSON from an LLM response. Returns raw text on failure."""
    try:
        cleaned = raw_response.strip()
        # Remove <think>...</think> blocks (for qwen3)
        cleaned = re.sub(r'<think>.*?</think>', '', cleaned, flags=re.DOTALL).strip()
        # Remove ```json ... ``` wrappers
        if cleaned.startswith('```'):
            cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
            cleaned = re.sub(r'\s*```$', '', cleaned)
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {
            'match_score': 0,
            'decision': 'Parse Error',
            'matching_skills': [],
            'missing_skills': [],
            'strengths': '',
            'weaknesses': '',
            'overall_assessment': f'Raw response: {raw_response[:500]}'
        }


def analyze_job_match_openai(cv_text: str, job_info: dict) -> dict:
    """Perform matching analysis using OpenAI GPT-4o-mini."""
    user_prompt = build_matching_user_prompt(cv_text, job_info)

    try:
        response = openai_client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': JOB_MATCHING_SYSTEM_PROMPT},
                {'role': 'user', 'content': user_prompt}
            ],
            temperature=0.3,
            max_tokens=1000,
        )
        raw = response.choices[0].message.content
        result = parse_llm_response(raw)
        result['model'] = 'GPT-4o-mini'
        return result
    except Exception as e:
        return {
            'model': 'GPT-4o-mini',
            'match_score': 0,
            'decision': 'Error',
            'matching_skills': [],
            'missing_skills': [],
            'strengths': '',
            'weaknesses': '',
            'overall_assessment': f'Error: {str(e)}'
        }


def analyze_job_match_ollama(cv_text: str, job_info: dict) -> dict:
    """Perform matching analysis using Ollama (llama3.2:1b)."""
    user_prompt = build_matching_user_prompt(cv_text, job_info)

    try:
        response = ollama_client.chat.completions.create(
            model='llama3.2:1b',
            messages=[
                {'role': 'system', 'content': JOB_MATCHING_SYSTEM_PROMPT},
                {'role': 'user', 'content': user_prompt}
            ],
            temperature=0.3,
        )
        raw = response.choices[0].message.content
        result = parse_llm_response(raw)
        result['model'] = 'Ollama llama3.2:1b'
        return result
    except Exception as e:
        return {
            'model': 'Ollama llama3.2:1b',
            'match_score': 0,
            'decision': 'Error',
            'matching_skills': [],
            'missing_skills': [],
            'strengths': '',
            'weaknesses': '',
            'overall_assessment': f'Error: {str(e)}'
        }


print("Matching functions defined successfully.")

Matching functions defined successfully.


## 6. Main Workflow

Scrape listings from HiringCafe and analyze each one with both OpenAI and Ollama.

**Settings**: Adjust the variables below to your needs.

In [6]:
# ═══════════════════════════════════════════════════
# SETTINGS - Set your desired category and job count
# ═══════════════════════════════════════════════════

CATEGORY_URL = "https://hiringcafe.com/jobs/software-engineer"  # Category to scan
MAX_JOBS = 5  # Maximum number of listings to analyze

# ═══════════════════════════════════════════════════

print("Job matching analysis starting...")
print(f"   Category: {CATEGORY_URL}")
print(f"   Max listings: {MAX_JOBS}")
print(f"{'=' * 60}\n")

# 1. Fetch job listing links
job_links = fetch_job_links(CATEGORY_URL, max_jobs=MAX_JOBS)

if not job_links:
    print("No listings found! Please check the URL.")
else:
    # 2. For each listing, fetch details and run analysis
    all_results = []

    for i, job_link in enumerate(job_links, 1):
        print(f"\n{'─' * 60}")
        print(f"[{i}/{len(job_links)}] {job_link['title']}")
        print(f"   URL: {job_link['url']}")

        # Fetch listing details
        try:
            job_details = fetch_job_details(job_link['url'])
            print(f"   Listing details fetched ({len(job_details['raw_text'])} chars)")
        except Exception as e:
            print(f"   Failed to fetch listing details: {e}")
            continue

        # OpenAI analysis
        print(f"   OpenAI analyzing...")
        openai_result = analyze_job_match_openai(cv_text, job_details)
        print(f"   OpenAI: {openai_result.get('decision', '?')} (Score: {openai_result.get('match_score', '?')}/10)")

        # Ollama analysis
        print(f"   Ollama analyzing...")
        ollama_result = analyze_job_match_ollama(cv_text, job_details)
        print(f"   Ollama: {ollama_result.get('decision', '?')} (Score: {ollama_result.get('match_score', '?')}/10)")

        all_results.append({
            'job': job_details,
            'openai': openai_result,
            'ollama': ollama_result
        })

        # Rate limiting - avoid overloading APIs
        time.sleep(1)

    print(f"\n{'=' * 60}")
    print(f"Analysis complete! {len(all_results)} listings evaluated.")

Job matching analysis starting...
   Category: https://hiringcafe.com/jobs/software-engineer
   Max listings: 5

Scanning listings: https://hiringcafe.com/jobs/software-engineer
   Found 5 listings

────────────────────────────────────────────────────────────
[1/5] Software Engineer Software Developer Altamira Technologies Fairborn
   URL: https://hiringcafe.com/job/software-engineer-software-developer-altamira-technologies-fairborn-iamcidcqmh8iwhh0
   Listing details fetched (2216 chars)
   OpenAI analyzing...
   OpenAI: Not a Match (Score: 3/10)
   Ollama analyzing...
   Ollama: Good Match (Score: 8/10)

────────────────────────────────────────────────────────────
[2/5] Software Engineer Lead Software Engineer State Farm Bloomington
   URL: https://hiringcafe.com/job/software-engineer-lead-software-engineer-state-farm-bloomington-kac5xufceyhyds01
   Listing details fetched (5000 chars)
   OpenAI analyzing...
   OpenAI: Partial Match (Score: 6/10)
   Ollama analyzing...
   Ollama: Par

## 7. Comparative Results Report

Displays OpenAI and Ollama results side by side for each listing.

In [8]:
def score_bar(score: int, max_score: int = 10) -> str:
    """Create a visual emoji bar for the score."""
    try:
        score = int(score)
    except (ValueError, TypeError):
        return "⬜" * max_score
    filled = min(score, max_score)
    empty = max_score - filled
    if score >= 7:
        char = "🟩"
    elif score >= 4:
        char = "🟨"
    else:
        char = "🟥"
    return char * filled + "⬜" * empty


def format_result_card(result: dict) -> str:
    """Create a comparative result card for a single listing."""
    job = result['job']
    openai_r = result['openai']
    ollama_r = result['ollama']

    # Decision emojis
    decision_emoji = {
        'Good Match': '✅',
        'Partial Match': '⚠️',
        'Not a Match': '❌',
    }

    openai_emoji = decision_emoji.get(openai_r.get('decision', ''), '❓')
    ollama_emoji = decision_emoji.get(ollama_r.get('decision', ''), '❓')

    # Format matching and missing skills
    openai_match = ', '.join(openai_r.get('matching_skills', [])[:5]) or 'Not specified'
    openai_miss = ', '.join(openai_r.get('missing_skills', [])[:5]) or 'None'
    ollama_match = ', '.join(ollama_r.get('matching_skills', [])[:5]) or 'Not specified'
    ollama_miss = ', '.join(ollama_r.get('missing_skills', [])[:5]) or 'None'

    md = f"""
---

### 🏢 {job['title']}
**Company:** {job['company']} | **Link:** [{job['url'].split('/job/')[1][:40]}...]({job['url']})

| | 🤖 **OpenAI GPT-4o-mini** | 🦙 **Ollama qwen3:4b** |
|---|---|---|
| **Decision** | {openai_emoji} {openai_r.get('decision', 'N/A')} | {ollama_emoji} {ollama_r.get('decision', 'N/A')} |
| **Score** | {score_bar(openai_r.get('match_score', 0))} **{openai_r.get('match_score', 0)}/10** | {score_bar(ollama_r.get('match_score', 0))} **{ollama_r.get('match_score', 0)}/10** |
| **Matching Skills** | {openai_match} | {ollama_match} |
| **Missing Skills** | {openai_miss} | {ollama_miss} |

**🤖 OpenAI Assessment:** {openai_r.get('overall_assessment', 'N/A')}

**🦙 Ollama Assessment:** {ollama_r.get('overall_assessment', 'N/A')}
"""
    return md


# ═══════════════════════════════════════════════════
# Display results
# ═══════════════════════════════════════════════════

if all_results:
    # Summary table
    summary_md = "# Job Matching Results\n\n"
    summary_md += "## Summary Table\n\n"
    summary_md += "| # | Position | Company | OpenAI Score | Ollama Score | OpenAI Decision | Ollama Decision |\n"
    summary_md += "|---|---------|--------|-------------|-------------|----------------|----------------|\n"

    for i, r in enumerate(all_results, 1):
        oi_score = r['openai'].get('match_score', 0)
        ol_score = r['ollama'].get('match_score', 0)
        oi_decision = r['openai'].get('decision', 'N/A')
        ol_decision = r['ollama'].get('decision', 'N/A')
        title_short = r['job']['title'][:35]
        company_short = r['job']['company'][:20]
        summary_md += f"| {i} | {title_short} | {company_short} | **{oi_score}/10** | **{ol_score}/10** | {oi_decision} | {ol_decision} |\n"

    # Detailed cards
    summary_md += "\n## Detailed Analysis\n"
    for r in all_results:
        summary_md += format_result_card(r)

    # Model comparison statistics
    openai_scores = [r['openai'].get('match_score', 0) for r in all_results]
    ollama_scores = [r['ollama'].get('match_score', 0) for r in all_results]

    # Convert to int (some models may return strings)
    openai_scores = [int(s) if isinstance(s, (int, float, str)) and str(s).isdigit() else 0 for s in openai_scores]
    ollama_scores = [int(s) if isinstance(s, (int, float, str)) and str(s).isdigit() else 0 for s in ollama_scores]

    if openai_scores and ollama_scores:
        avg_openai = sum(openai_scores) / len(openai_scores)
        avg_ollama = sum(ollama_scores) / len(ollama_scores)

        summary_md += f"""\n\n---\n
## Model Comparison\n
| Metric | 🤖 OpenAI GPT-4o-mini | 🦙 Ollama qwen3:4b |
|--------|----------------------|--------------------|
| **Average Score** | {avg_openai:.1f}/10 | {avg_ollama:.1f}/10 |
| **Highest Score** | {max(openai_scores)}/10 | {max(ollama_scores)}/10 |
| **Lowest Score** | {min(openai_scores)}/10 | {min(ollama_scores)}/10 |
"""

    display(Markdown(summary_md))
else:
    print("No results found. Make sure you ran the previous cell first.")

# Job Matching Results

## Summary Table

| # | Position | Company | OpenAI Score | Ollama Score | OpenAI Decision | Ollama Decision |
|---|---------|--------|-------------|-------------|----------------|----------------|
| 1 | Software Engineer/Software Develope | Altamira Technologie | **3/10** | **8/10** | Not a Match | Good Match |
| 2 | Software Engineer/Lead Software Eng | State Farm | **6/10** | **0/10** | Partial Match | Parse Error |
| 3 | Software Engineer at World Travel H | World Travel Holding | **3/10** | **0/10** | Not a Match | Parse Error |
| 4 | Engineer Software/Principal Enginee | Northrop Grumman | **5/10** | **8/10** | Partial Match | Good Match |
| 5 | Software Engineer at Rabot | Rabot | **8/10** | **0/10** | Good Match | Parse Error |

## Detailed Analysis

---

### 🏢 Software Engineer/Software Developer at Altamira Technologies
**Company:** Altamira Technologies | **Link:** [software-engineer-software-developer-alt...](https://hiringcafe.com/job/software-engineer-software-developer-altamira-technologies-fairborn-iamcidcqmh8iwhh0)

| | 🤖 **OpenAI GPT-4o-mini** | 🦙 **Ollama qwen3:4b** |
|---|---|---|
| **Decision** | ❌ Not a Match | ✅ Good Match |
| **Score** | 🟥🟥🟥⬜⬜⬜⬜⬜⬜⬜ **3/10** | 🟩🟩🟩🟩🟩🟩🟩🟩⬜⬜ **8/10** |
| **Matching Skills** | Python, Git, Docker, Cloud Computing | Python, Git, Docker, Cloud Computing, Image Processing |
| **Missing Skills** | 7+ years software development experience, image processing, TS/SCI clearance, experience leading end-to-end design, Bachelor’s degree in CS/CE/Engineering or related field | None |

**🤖 OpenAI Assessment:** While the candidate possesses relevant technical skills and a solid foundation in software engineering, they fall significantly short of the experience requirements and specific domain expertise outlined in the job listing. This makes them unsuitable for the position at Altamira Technologies.

**🦙 Ollama Assessment:** The candidate's technical skills and experience align well with the job requirements, making them a strong fit for this position.

---

### 🏢 Software Engineer/Lead Software Engineer at State Farm
**Company:** State Farm | **Link:** [software-engineer-lead-software-engineer...](https://hiringcafe.com/job/software-engineer-lead-software-engineer-state-farm-bloomington-kac5xufceyhyds01)

| | 🤖 **OpenAI GPT-4o-mini** | 🦙 **Ollama qwen3:4b** |
|---|---|---|
| **Decision** | ⚠️ Partial Match | ❓ Parse Error |
| **Score** | 🟨🟨🟨🟨🟨🟨⬜⬜⬜⬜ **6/10** | ⬜⬜⬜⬜⬜⬜⬜⬜⬜⬜ **0/10** |
| **Matching Skills** | Python, AWS, APIs, Docker, Kubernetes | Not specified |
| **Missing Skills** | Terraform, JSON, React, Cloud Architecture, DevSecOps | None |

**🤖 OpenAI Assessment:** While the candidate possesses relevant skills in Python and AWS, they fall short in several key areas required for the position, such as Terraform and React. Their experience in backend development and machine learning is a plus, but they would need to enhance their cloud architecture and DevSecOps knowledge to be a stronger contender for this role.

**🦙 Ollama Assessment:** Raw response: {
    "match_score": 7,
    "decision": "Good Match",
    "matching_skills": ["Python", "AWS (Lambda, EC2, SQS, S3, Cloudfront, API Gateway, SNS, Sagemaker)"],
    "missing_skills": [],
    "strengths": "The candidate has a strong background in Python and AWS technologies, which aligns well with the job requirements. They also have experience in cloud architecture and development, which is relevant to the position.",
    "weaknesses": "While the candidate has some experience in software delivery

---

### 🏢 Software Engineer at World Travel Holdings
**Company:** World Travel Holdings | **Link:** [software-engineer-world-travel-holdings-...](https://hiringcafe.com/job/software-engineer-world-travel-holdings-los-angeles-california-rxmk7e754jaz79dg)

| | 🤖 **OpenAI GPT-4o-mini** | 🦙 **Ollama qwen3:4b** |
|---|---|---|
| **Decision** | ❌ Not a Match | ❓ Parse Error |
| **Score** | 🟥🟥🟥⬜⬜⬜⬜⬜⬜⬜ **3/10** | ⬜⬜⬜⬜⬜⬜⬜⬜⬜⬜ **0/10** |
| **Matching Skills** | Python, REST APIs, Git, CI/CD | Not specified |
| **Missing Skills** | .NET Core/6+, C#, Angular, React, Vue.js | None |

**🤖 OpenAI Assessment:** While the candidate has relevant backend development experience and a strong foundation in software engineering, they do not meet the essential requirements for this position, particularly in .NET and frontend technologies. The lack of experience with the specified tools and frameworks makes them unsuitable for this role.

**🦙 Ollama Assessment:** Raw response: {
    "match_score": 8,
    "decision": "Good Match",
    "matching_skills": [".NET", "C#", "Angular", "React", "Vue.js", "HTML5", "CSS3", "TypeScript", "MS SQL Server", "MySQL", "REST", "JSON", "XML", "SOAP", "CI/CD", "DevOps", "Git"],
    "missing_skills": [".NET Core/6+", "Frontend", "Database Management", "Webhooks & Event-Driven Design", "Data Consumption", "Engineering Excellence (Automation)"],
    "strengths": "Strong .NET, frontend, databases, and web services experience; excellent comm

---

### 🏢 Engineer Software/Principal Engineer Software at Northrop Grumman
**Company:** Northrop Grumman | **Link:** [engineer-software-principal-engineer-sof...](https://hiringcafe.com/job/engineer-software-principal-engineer-software-northrop-grumman-e38aru6ymv3txogx)

| | 🤖 **OpenAI GPT-4o-mini** | 🦙 **Ollama qwen3:4b** |
|---|---|---|
| **Decision** | ⚠️ Partial Match | ✅ Good Match |
| **Score** | 🟨🟨🟨🟨🟨⬜⬜⬜⬜⬜ **5/10** | 🟩🟩🟩🟩🟩🟩🟩🟩⬜⬜ **8/10** |
| **Matching Skills** | Python, Java, Docker, Kubernetes, Selenium | Software Engineering, AI/ML Enthusiast, Object-Oriented Programming Languages (C/C++) |
| **Missing Skills** | C/C++, RTOS, Windows, Ansible, Experience with safety-critical aviation systems | Cloud Computing, Databases (PostgreSQL, MongoDB) |

**🤖 OpenAI Assessment:** While the candidate possesses a solid foundation in software engineering and relevant technical skills, they do not fulfill several key requirements for the position, particularly in terms of language proficiency, experience with safety-critical systems, and security clearance. Therefore, they are a partial match for the role.

**🦙 Ollama Assessment:** Based on the analysis, it appears that the candidate has a good fit for this role due to their technical background and experience. However, they may need to acquire additional skills or knowledge in areas such as cloud computing and databases to fully meet the requirements of the position.

---

### 🏢 Software Engineer at Rabot
**Company:** Rabot | **Link:** [software-engineer-rabot-arlington-texas-...](https://hiringcafe.com/job/software-engineer-rabot-arlington-texas-x4qvnjkk6syxui72)

| | 🤖 **OpenAI GPT-4o-mini** | 🦙 **Ollama qwen3:4b** |
|---|---|---|
| **Decision** | ✅ Good Match | ❓ Parse Error |
| **Score** | 🟩🟩🟩🟩🟩🟩🟩🟩⬜⬜ **8/10** | ⬜⬜⬜⬜⬜⬜⬜⬜⬜⬜ **0/10** |
| **Matching Skills** | Python, FastAPI, AWS, Docker, Kubernetes | Not specified |
| **Missing Skills** | Frontend development, Experience with real-time data or video/image pipelines, Experience building B2B SaaS products | None |

**🤖 OpenAI Assessment:** Alperen Kurt is a strong candidate for the Software Engineer position at Rabot, demonstrating relevant experience in backend development and cloud infrastructure. However, to fully meet the job requirements, he should enhance his frontend development skills and gain experience in real-time data handling and B2B SaaS product development.

**🦙 Ollama Assessment:** Raw response: {
    "match_score": 8,
    "decision": "Good Match",
    "matching_skills": ["Software Engineering", "AI/ML Enthusiast", "Full Stack Development", "Cloud Infrastructure"],
    "missing_skills": [],
    "strengths": "3+ years of professional software engineering experience, shipped production software that real users depend on, strong across the stack, comfortable with cloud infrastructure",
    "weaknesses": "Not less about where you worked and more about what you've built, not perfect in the b


---

## Model Comparison

| Metric | 🤖 OpenAI GPT-4o-mini | 🦙 Ollama qwen3:4b |
|--------|----------------------|--------------------|
| **Average Score** | 5.0/10 | 3.2/10 |
| **Highest Score** | 8/10 | 8/10 |
| **Lowest Score** | 3/10 | 0/10 |
